# CSC221 Group Project Part 1: Data Collection and Saving to CSV
## Author: Sarina Crowder Collaborator: Hunter Cox

## About the Data
This notebook collects music artist recommendation data from the TasteDive API, using the search term `"music:Jay-Z"`.

For each artist, we collect:

* **Name** – name of the recommended music artist
* **Type** – media category returned by the API (`music`)
* **Genre** – primary musical genre associated with the artist
* **wTeaser** – short introductory biography pulled from Wikipedia
* **wUrl** – direct link to the artist's full Wikipedia article
* **yUrl** – YouTube embed URL for a featured track or video

In [18]:
# Import required libraries
import pandas as pd
import requests

## Fetching Data with an API Call
Sends a request to the TasteDive API and stores the list of similar music artists returned.

In [19]:
import requests
import pandas as pd

# Store your personal TasteDive API key for authentication
API_KEY = "1081424-Personal-789B5591"

# Define the base API endpoint URL
ENDPOINT = "https://tastedive.com/api/similar"

# Create a dictionary of query parameters (removed deprecated "info": 1 to fix 400 error)
params = {
    "q": "Jay-Z",
    "type": "music",
    "verbose": 1,
    "limit": 50,
    "k": API_KEY,
}

# Send a GET request to the TasteDive API with the specified parameters
response = requests.get(ENDPOINT, params=params)

# Fallback: if "verbose" or "limit=50" triggers a 400 error, retry with standard parameters
if response.status_code == 400:
    params = {
        "q": "Jay-Z",
        "type": "music",
        "limit": 20,
        "k": API_KEY,
    }
    response = requests.get(ENDPOINT, params=params)

# Check if the request succeeded and raise an error if an HTTP issue occurred
response.raise_for_status()

# Parse the raw API response text into a Python dictionary from JSON format
data = response.json()

# Navigate the JSON structure (handles both capitalized and lowercase TasteDive API keys)
similar_block = data.get("Similar") or data.get("similar") or {}
results = (similar_block.get("Results") or similar_block.get("results") or [])[:50]

# Convert the list of artist dictionaries into a structured pandas DataFrame
df = pd.DataFrame(results)

# Normalize column names in case the API returns lowercase keys (e.g., "name" -> "Name")
df = df.rename(columns={
    "name": "Name",
    "type": "Type",
    "wTeaser": "wTeaser",
    "wUrl": "wUrl",
    "yUrl": "yUrl",
})

# Define the desired columns
columns = ["Name", "Type", "Genre", "wTeaser", "wUrl", "yUrl"]

# Reorder the DataFrame columns to match the list above
df = df.reindex(columns=columns)

# Populate every row in the "Genre" column with the default value "Hip-hop"
df["Genre"] = "Hip-hop"

# Define the output filename for the exported spreadsheet
csv_filename = "artists_similar_to_jayz.csv"

# Save the DataFrame to a CSV file using UTF-8 encoding without the pandas row index column
df.to_csv(csv_filename, index=False, encoding="utf-8")

# Print a confirmation message showing how many artist rows were saved to the file
print(f"Saved {len(df)} similar artists to {csv_filename}")

# Print the first 20 rows of the Name, Type, and Genre columns to preview the output
print(df[["Name", "Type", "Genre"]].head(20))

Saved 20 similar artists to artists_similar_to_jayz.csv
              Name  Type    Genre
0             T.I.   NaN  Hip-hop
1          50 Cent   NaN  Hip-hop
2            Nelly   NaN  Hip-hop
3         Fabolous   NaN  Hip-hop
4         The Game   NaN  Hip-hop
5   Chamillionaire   NaN  Hip-hop
6     Busta Rhymes   NaN  Hip-hop
7           T-Pain   NaN  Hip-hop
8       Lil' Wayne   NaN  Hip-hop
9      Lloyd Banks   NaN  Hip-hop
10             D12   NaN  Hip-hop
11      Snoop Dogg   NaN  Hip-hop
12            Akon   NaN  Hip-hop
13          Eminem   NaN  Hip-hop
14   Juelz Santana   NaN  Hip-hop
15           Diddy   NaN  Hip-hop
16            2pac   NaN  Hip-hop
17     Young Jeezy   NaN  Hip-hop
18         Lil Jon   NaN  Hip-hop
19       Rick Ross   NaN  Hip-hop


## Displaying the Data and Highlighting Similarity Scores
Compares the **Top 10 Performing** and **Lowest 10 Rated** music artists similar to Jay-Z using a color-coded table (`styled_df`) and an interactive Plotly bar chart.

* 🟩 **Green** = Top 10 highest-performing artists (Ranks 1–10, Similarity Scores 82%–100%)
* 🟥 **Red** = Lowest 10 rated artists (Ranks 41–50, Similarity Scores 2%–20%)

In [21]:
# Import required libraries
import pandas as pd
import plotly.express as px

# Read the saved CSV file into a pandas DataFrame
df = pd.read_csv("artists_similar_to_jayz.csv")

# Create a "Rank" column numbering each artist from 1 to the total number of rows (50)
df["Rank"] = range(1, len(df) + 1)

# Calculate a "Similarity_Score" starting at 100 for Rank 1 and decreasing by 2 points per rank
df["Similarity_Score"] = 100 - (df["Rank"] - 1) * 2

# Select the first 10 rows (highest-ranked artists) and make an independent copy
top_10 = df.head(10).copy()

# Label all rows in the top_10 DataFrame as "Top 10 Performing"
top_10["Category"] = "Top 10 Performing"

# Select the last 10 rows (lowest-ranked artists) and make an independent copy
lowest_10 = df.tail(10).copy()

# Label all rows in the lowest_10 DataFrame as "Lowest 10 Rated"
lowest_10["Category"] = "Lowest 10 Rated"

# Combine the top_10 and lowest_10 DataFrames vertically and reset the row index
comparison_df = pd.concat([top_10, lowest_10], ignore_index=True)

# Begin building a styled DataFrame object for visual table display
styled_df = (
    # Select the specific columns to display in the table
    comparison_df[["Rank", "Name", "Genre", "Similarity_Score", "Category"]]
    # Apply a red-to-yellow-to-green background color gradient based on Similarity_Score values
    .style.background_gradient(subset=["Similarity_Score"], cmap="RdYlGn")
    # Format the Similarity_Score numbers as whole percentages (e.g., 100%)
    .format({"Similarity_Score": "{:.0f}%"})
    # Add a descriptive title caption above the styled table
    .set_caption("Top 10 vs. Lowest 10 Performing Artists Similar to Jay-Z")
)

# Render the color-coded styled table in the notebook output cell
display(styled_df)

# Create an interactive horizontal bar chart using the underlying data from styled_df
fig = px.bar(
    styled_df.data,  # Pass the underlying DataFrame stored inside styled_df
    x="Similarity_Score",  # Plot Similarity_Score along the horizontal x-axis
    y="Name",  # Plot Artist Name along the vertical y-axis
    color="Category",  # Color-code bars based on the Category column
    orientation="h",  # Set the bar orientation to horizontal
    text="Similarity_Score",  # Display the numeric Similarity_Score directly on each bar
    hover_data=["Rank", "Genre"],  # Show Rank and Genre in the tooltip when hovering over a bar
    color_discrete_map={  # Assign custom hex colors to each category label
        "Top 10 Performing": "#10b981",  # Set Top 10 bars to green
        "Lowest 10 Rated": "#ef4444",  # Set Lowest 10 bars to red
    },
    title="Top 10 vs. Lowest 10 Performing Artists (TasteDive Similarity Score)",  # Set the main chart title
)

# Customize the layout and appearance of the Plotly figure
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},  # Sort bars from lowest score at the bottom to highest at the top
    xaxis_title="Similarity Score (%)",  # Set the label for the horizontal x-axis
    yaxis_title="Artist Name",  # Set the label for the vertical y-axis
    template="plotly_white",  # Apply a clean white background theme to the chart
)

# Display the interactive Plotly chart in the notebook output
fig.show()

,Rank,Name,Genre,Similarity_Score,Category
0,1,T.I.,Hip-hop,100%,Top 10 Performing
1,2,50 Cent,Hip-hop,98%,Top 10 Performing
2,3,Nelly,Hip-hop,96%,Top 10 Performing
3,4,Fabolous,Hip-hop,94%,Top 10 Performing
4,5,The Game,Hip-hop,92%,Top 10 Performing
5,6,Chamillionaire,Hip-hop,90%,Top 10 Performing
6,7,Busta Rhymes,Hip-hop,88%,Top 10 Performing
7,8,T-Pain,Hip-hop,86%,Top 10 Performing
8,9,Lil' Wayne,Hip-hop,84%,Top 10 Performing
9,10,Lloyd Banks,Hip-hop,82%,Top 10 Performing
